In [2]:
! pip install -q torch-adopt optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 97.8 MB/s eta 0:00:00


### Hyperparameter Optimization

In [1]:
from typing import Any, Callable, Dict, Literal, Tuple

import optuna
import torch
import torch.optim as optim
import torch.utils.data as data
from adopt import ADOPT
from torch import nn


OptimizerName = Literal["Adam", "AdamW", "ADOPT", "RMSprop", "SGD"]
ModelFactory = Callable[..., nn.Module]
DataLoaderFactory = Callable[..., Tuple[data.DataLoader, data.DataLoader, data.DataLoader]]
TrainEvalLoop = Callable[..., float]


class ParamOptimizer:
    def __init__(self):
        self.results = {}

    def _get_optimizer_search_space(
        self,
        optimizer_name: OptimizerName,
        trial: optuna.Trial,
    ) -> Dict[str, Any]:
        """Define search spaces for each optimizer."""

        if optimizer_name == 'ADOPT':
            return {
                'lr': trial.suggest_float('lr', 1e-5, 1e-2, log=True),
                'betas': (
                    trial.suggest_float('beta1', 0.8, 0.99),
                    trial.suggest_float('beta2', 0.9, 0.9999)
                ),
                'weight_decay': trial.suggest_float('weight_decay', 0.0, 0.1, log=True),
                'decouple': True,
            }

        elif optimizer_name == 'Adam':
            return {
                'lr': trial.suggest_float('lr', 1e-5, 1e-2, log=True),
                'betas': (
                    trial.suggest_float('beta1', 0.8, 0.99),
                    trial.suggest_float('beta2', 0.9, 0.999)
                ),
                'weight_decay': trial.suggest_float('weight_decay', 0.0, 1e-2, log=True),
            }

        elif optimizer_name == 'AdamW':
            return {
                'lr': trial.suggest_float('lr', 1e-5, 1e-2, log=True),
                'betas': (
                    trial.suggest_float('beta1', 0.8, 0.99),
                    trial.suggest_float('beta2', 0.9, 0.999)
                ),
                'weight_decay': trial.suggest_float('weight_decay', 0.0, 0.1, log=True),
            }

        elif optimizer_name == 'RMSprop':
            return {
                'lr': trial.suggest_float('lr', 1e-5, 1e-2, log=True),
                'alpha': trial.suggest_float('alpha', 0.9, 0.999),
                'weight_decay': trial.suggest_float('weight_decay', 0.0, 1e-2, log=True),
                'momentum': trial.suggest_float('momentum', 0.0, 0.1),
            }

        elif optimizer_name == 'SGD':
            return {
                'lr': trial.suggest_float('lr', 1e-5, 1e-2, log=True),
                'momentum': trial.suggest_float('momentum', 0.8, 0.99),
                'weight_decay': trial.suggest_float('weight_decay', 0.0, 1e-2, log=True),
            }

        else:
            raise ValueError(f"Unknown optimizer: {optimizer_name}")

    def _create_optimizer(
        self,
        optimizer_name: str,
        model: nn.Module,
        **kwargs,
    ) -> optim.Optimizer:
        """Create optimizer instance with given parameters."""

        model_params = model.parameters()

        if optimizer_name == 'SGD':
            return optim.SGD(model_params, **kwargs)
        elif optimizer_name == 'RMSprop':
            return optim.RMSprop(model_params, **kwargs)
        elif optimizer_name == 'Adam':
            return optim.Adam(model_params, **kwargs)
        elif optimizer_name == 'AdamW':
            return optim.AdamW(model_params, **kwargs)
        elif optimizer_name in ['ADOPT']:
            return ADOPT(model_params, **kwargs)
        else:
            raise ValueError(f"Unknown optimizer: {optimizer_name}")

    def _objective_factory(
        self,
        optimizer_name: OptimizerName,
        model_factory: ModelFactory,
        dataloader_factory: DataLoaderFactory,
        train_eval_loop: TrainEvalLoop,
        device: str = "cuda",
    ) -> Callable[[optuna.Trial], float]:
        """Create objective function for a specific optimizer."""

        def objective(trial: optuna.Trial) -> float:
            optimizer_params = self._get_optimizer_search_space(optimizer_name, trial)
            model = model_factory().to(device)
            optimizer = self._create_optimizer(optimizer_name, model, **optimizer_params)
            train_loader, val_loader, _ = dataloader_factory()

            val_accuracy = train_eval_loop(
                model=model,
                optimizer=optimizer,
                train_loader=train_loader,
                val_loader=val_loader,
                trial=trial,
                device=device,
            )

            return val_accuracy

        return objective

    def optimize_optimizer_params(
        self,
        optimizer_name: OptimizerName,
        model_factory: ModelFactory,
        dataloader_factory: DataLoaderFactory,
        train_eval_loop: TrainEvalLoop,
        *,
        n_trials: int = 20,
        n_startup_trials: int = 5,
        n_warmup_steps: int = 3,
        interval_steps: int = 2,
        timeout: int | None = None,
        device: str = "cuda",
    ) -> optuna.Study:
        """Optimize hyperparameters for a specific optimizer on a single task."""

        print(f"\nOptimizing {optimizer_name}...")

        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=42),
            pruner=optuna.pruners.MedianPruner(
                n_startup_trials=n_startup_trials,
                n_warmup_steps=n_warmup_steps,
                interval_steps=interval_steps,
            )
        )

        objective = self._objective_factory(
            optimizer_name,
            model_factory,
            dataloader_factory,
            train_eval_loop,
            device,
        )
        study.optimize(objective, n_trials=n_trials, timeout=timeout)

        self.results[optimizer_name] = {
            'best_params': study.best_params,
            'best_value': study.best_value,
            'n_trials': len(study.trials),
            'study': study,
        }

        print(f"{optimizer_name} - Best value: {study.best_value:.4f}")
        print(f"{optimizer_name} - Best params: {study.best_params}")

        return study

    def get_best_params_dict(
        self,
        optimizer_name: OptimizerName | None = None
    ) -> Dict[str, Any]:
        """Get the best parameters for an optimizer."""

        if optimizer_name:
            return self.results[optimizer_name]
        else:
            return self.results

### Image Classification on CIFAR100

In [7]:
import torchvision
from torchvision import models, transforms, datasets

CIFAR100_ROOT = "/data/cifar100"

def cifar100_model_factory(num_classes: int = 100) -> nn.Module:
    model = models.resnet18(num_classes=num_classes)
    # modify first conv layer to avoid upscaling to 224x224
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model

def cifar100_dataloader_factory(
    batch_size: int = 32,
    seed: int = 42
) -> Tuple[data.DataLoader, data.DataLoader, data.DataLoader]:
    # standard CIFAR-100 mean and std
    mean = [0.5071, 0.4865, 0.4409]
    std = [0.2673, 0.2564, 0.2761]

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    temp_dataset = datasets.CIFAR100(root=CIFAR100_ROOT, train=True, transform=transform, download=True)
    test_dataset = datasets.CIFAR100(root=CIFAR100_ROOT, train=False, transform=transform, download=True)

    train_size = int(0.8 * len(temp_dataset))
    val_size = len(temp_dataset) - train_size
    generator = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset = data.random_split(temp_dataset, [train_size, val_size], generator=generator)

    train_loader = data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

In [8]:
from sklearn.metrics import f1_score

def cifar100_train_eval_loop(
    *,
    model: nn.Module,
    optimizer: optim.Optimizer,
    train_loader: data.DataLoader,
    val_loader: data.DataLoader,
    trial: optuna.Trial | None = None,
    criterion: nn.Module = nn.CrossEntropyLoss(),
    epochs: int = 10,
    device: str = "cuda",
) -> float:
    model = model.to(device)

    for epoch in range(epochs):
        model.train()
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            loss.backward()
            optimizer.step()

        model.eval()
        all_preds = []
        all_targets = []
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                preds = torch.argmax(outputs, dim=1)
                all_preds.extend(preds.cpu().tolist())
                all_targets.extend(targets.cpu().tolist())

        f1 = f1_score(all_targets, all_preds, average="macro")
        print(f"Epoch {epoch+1}/{epochs} — F1 Score: {f1:.4f}")

        if trial:
            trial.report(f1, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

    return f1

#### Test factory functions

In [9]:
model = cifar100_model_factory()
optimizer = optim.Adam(model.parameters())

In [10]:
train_loader, val_loader, test_loader = cifar100_dataloader_factory()

In [11]:
final_f1 = cifar100_train_eval_loop(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    device="cuda",
)

Epoch 1/10 — F1 Score: 0.1643
Epoch 2/10 — F1 Score: 0.3289
Epoch 3/10 — F1 Score: 0.3885
Epoch 4/10 — F1 Score: 0.4651
Epoch 5/10 — F1 Score: 0.4748
Epoch 6/10 — F1 Score: 0.5028
Epoch 7/10 — F1 Score: 0.5075
Epoch 8/10 — F1 Score: 0.5161
Epoch 9/10 — F1 Score: 0.5093
Epoch 10/10 — F1 Score: 0.5046
